# SA-Jev · 公开 LightRAG 索引与数据准备
默认使用三套公开 N500 图，不对训练段落重新付费建图。产物保存到 MyDrive/SA-Jev/public_lightrag_v1/pilot 或 full；旧 full 目录保留。full 表示完整研究协议：8,000+4,000 道训练题、每数据集 500 道公开测试题，不表示 benchmark 全测试集。CPU 可运行，下载约 561 MB 索引和小型 MiniLM；不下载 OpenJev-4B。首次先跑 pilot；成功结束后默认刷新 Drive 并断开。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Use 'full' after the pilot; separate Drive directories.
AUTO_DISCONNECT = True  # Release Colab after successful preparation and Drive flush.
from uuid import uuid4
PREPARATION_RUN_ID = uuid4().hex
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py
SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [4]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/full/smoke/run_uwv32ui9
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471526622772, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 0.7853141460000188, 'eta_seconds': 0.7853141460000188, 'grad_norm': 0.0001167683003586717, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.6235347628593444, 'total_loss': 2.6235347628593444}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.693147212266922, 'rel': 0.6931336522102356, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 1.834144292000019, 'eta_seconds': 0.0, 'grad_norm': 0.0002577165432740003, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 9, 'task': 2.4412035405635835, 'total_loss': 2.4412035405635835, 'validation': {'rank': 1.7005923390388489, 'suf': 0.693146745363

{'test_only': True,
 'output_directory': '/content/drive/MyDrive/SA-Jev/full/smoke/run_uwv32ui9',
 'model': 'tiny causal encoder + actual upstream OpenJev DecisionHead/OptionSetInteractor',
 'data': 'synthetic fixtures',
 'device': 'cpu',
 'train_states': 6,
 'validation_states': 3,
 'preference_pairs': 6,
 'stage1_optimizer_steps': 2,
 'stage2_optimizer_steps': 2,
 'checkpoint_roundtrip': True,
 'evaluation_rows': 1,
 'evaluation_errors': 0,
 'peak_reserved_gib': 0.0,
 'not_verified': ['trained OpenJev-4B weights',
  'A100 80GB peak',
  'live LightRAG extraction',
  'official Jev API',
  'scientific performance']}

公开图的向量编码器为 all-MiniLM-L6-v2（384 维、256 token），与 BGE 不兼容。本地下载固定版本，仅用于查询和兼容性检查，不调用 embedding API。源发布未记录编码器 revision，加载图后用已存向量进行兼容性校验。

In [5]:
from sa_jev.embeddings import check_embeddings
embedding_check = await check_embeddings(config['retrieval'], ROOT/'embedding_check.json')
embedding_check

Loading local embedding: BAAI/bge-small-en-v1.5@5c38ec7c405ec4b44b94cc5a9bb96e735b38267a, device=cpu
Checking pinned BGE weight: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/model.safetensors
BGE weight cache is invalid: expected 133,466,304 bytes, got 79. Re-downloading the same revision once.
Preserved invalid cache entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/model.safetensors.invalid-f5445963


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

BGE weight verified: 133,466,304 bytes, SHA-256 matches.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Local embedding ready in 3.7s: dim=384, batch=16, max_length=512, normalized=True
Embedding check: (2, 384), finite=True, 0.18s, GPU peak reserved=0.000 GiB


{'backend': 'local',
 'model': 'BAAI/bge-small-en-v1.5',
 'revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a',
 'dimension': 384,
 'device': 'cpu',
 'load_seconds': 10.247464022999793,
 'encode_seconds': 0.18192570100018202,
 'texts_per_second': 16.490248400895258,
 'document_norms': [1.0, 1.0],
 'toy_top1_correct': True,
 'peak_gpu_reserved_gib': 0,
 'finite_vectors': True,
 'test_only': True}

下载并校验公开 ZIP，按重复条目分别导入三套图；2Wiki 使用包内完整冲突副本，详见 public_manifest.json。复用旧 full/raw 下载（若完整），否则下载原始训练源。仅采样训练问题的段落用于 BM25 难例，不调用建图 API。公开 500 题全部保留为测试；另从未用的 HippoRAG 问题族取验证与独立扰动题，不注入缺失支持证据。

In [ ]:
from sa_jev.public_indexes import download_public_assets, import_public_indexes, prepare_public_data
from sa_jev.preparation import download_sources
from sa_jev.io import read_json
archive = download_public_assets(PUBLIC_ASSETS)
index_manifest = import_public_indexes(archive, ROOT)
raw = ROOT/'raw'
legacy = DRIVE/'full/raw'
if MODE == 'full' and (legacy/'download_manifest.json').exists() and read_json(legacy/'download_manifest.json').get('formal'):
    raw = legacy
    print('Reusing existing source downloads:', raw)
download_sources(raw, limit=64 if MODE=='pilot' else None)
manifest = prepare_public_data(ROOT, PUBLIC_ASSETS, raw, config, mode=MODE)
manifest

在导入的三个图上执行 LightRAG 原生 mix（图＋向量）召回，缓存最大 K=256，再切片得到主实验 K=32。保留已发布图，不重新抽取文档；查询关键词提取仍会调用 .env 中的 DeepSeek。完整协议有 2,300 个查询（1,500 测试、200 验证、600 独立保留），pilot 18 个。初始缓存完成后，其余 notebook 复用候选。检查 public_pool_coverage.csv；缺失支持、未知标签仍掩蔽。

In [6]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

Extraction service: https://www.autodl.art/api/v1; model=DeepSeek-V4.1-Flash; credential file=/content/drive/MyDrive/SA-Jev/.env
Loading local embedding: BAAI/bge-small-en-v1.5@5c38ec7c405ec4b44b94cc5a9bb96e735b38267a, device=cpu
Checking pinned BGE weight: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/model.safetensors
BGE weight cache is invalid: expected 133,466,304 bytes, got 79. Re-downloading the same revision once.
Preserved invalid cache entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/model.safetensors.invalid-dd8626b3


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

BGE weight verified: 133,466,304 bytes, SHA-256 matches.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Local embedding ready in 3.3s: dim=384, batch=16, max_length=512, normalized=True


INFO: [] Created new empty graph file: /content/drive/MyDrive/SA-Jev/full/index/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 13874 KV load full_docs with 32 records
INFO: [] Process 13874 KV load text_chunks with 0 records
INFO: [] Process 13874 KV load full_entities with 0 records
INFO: [] Process 13874 KV load full_relations with 0 records
INFO: [] Process 13874 KV load entity_chunks with 0 records
INFO: [] Process 13874 KV load relation_chunks with 0 records
INFO: [] Process 13874 KV load llm_response_cache with 0 records
INFO: [] Process 13874 doc status load doc_status with 96 records


Indexing 657246 unique documents; completed documents will be reused
Resuming existing failed documents through LightRAG's native retry pipeline (one retry attempt; no duplicate insertion).


INFO: No documents to process
INFO: Manual retry 9b0387fe: reset 32 FAILED document(s) to PENDING (exclusive phase, no worker running)
INFO: Processing 32 document(s)
INFO: Parsing (native): e_dc3f9d85abbad1c9caca
INFO: Parsing (native): e_d280d1b96e3ac8da3a65
INFO: Parsing (native): e_968cfde06df5e3be2562
INFO: Parsing (native): e_9e71f11be3fe44d95c92
INFO: Parsing (native): e_ed7ca68ad4e32e4a8116
INFO: Parsing (native): e_393dd76e10d89367962b
INFO: Parsing (native): e_39425303ee55baba4bde
INFO: Parsing (native): e_84185bf776c1ed122b86
INFO: Parsing (native): e_bd2b712f92d01a331798
INFO: Parsing (native): e_08e6c9e841fbe34bf95f
INFO: Extracting stage 1/32: e_dc3f9d85abbad1c9caca
INFO: Processing d-id: e_dc3f9d85abbad1c9caca
INFO: Extracting stage 2/32: e_d280d1b96e3ac8da3a65
INFO: Processing d-id: e_d280d1b96e3ac8da3a65
INFO: [resume] e_dc3f9d85abbad1c9caca: purging 1 chunk(s) and associated KG entries from a previous run before rebuilding under current process_options
INFO: [resume] 

CancelledError: 

In [ ]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.preparation import finish_preparation
completion = finish_preparation(ROOT, config, mode=MODE, run_id=PREPARATION_RUN_ID)
completion

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。

结束后自动断开：只有本次运行已成功完成才执行。先刷新并卸载 Drive，再释放 Colab 运行时。Drive 刷新失败或流程失败时不主动释放；重新连接需重装环境并挂载 Drive，已保存数据保留。本地运行会跳过此步骤。

In [ ]:
from sa_jev.environment import disconnect_after_preparation
disconnect_after_preparation(ROOT, run_id=PREPARATION_RUN_ID, enabled=AUTO_DISCONNECT)